# music-preference-final



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'bryn8tbeac1s15'
os.environ['DataZoneDomainId'] = 'dzd-advo2csjkd84mh'
os.environ['DataZoneEnvironmentId'] = '5csb0f86mi6949'
os.environ['DataZoneDomainRegion'] = 'us-east-2'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "bryn8tbeac1s15",
                "DataZoneDomainId": "dzd-advo2csjkd84mh",
                "DataZoneEnvironmentId": "5csb0f86mi6949",
                "DataZoneDomainRegion": "us-east-2",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
from sagemaker.sklearn.estimator import SKLearn
import sagemaker

role = sagemaker.get_execution_role()

from sagemaker.sklearn.model import SKLearnModel

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [0]:
estimator = SKLearn(
    entry_point='script.py',
    role=role,
    instance_type='ml.c5.2xlarge',
    instance_count=1,
    framework_version='1.2-1',   # any recent version is fine
    py_version='py3',
    dependencies=['requirements.txt'],
    output_path='s3://news-aggregator-sadrian-bucket/models/',
    hyperparameters={
        'early_stopping_rounds': 50,
        'learning_rate': 0.05,
        'l2': 2.0,
        'num_leaves': 64,
        'max_depth': -1,
        'min_data_in_leaf': 100,
        'training_path': 's3://music-preference-bucket/train/',
    }
)



In [0]:
model_s3_path = "s3://amazon-sagemaker-134218838914-us-east-2-4oxmckbtrzadyh/models/model.tar.gz"

predictor = SKLearnModel(
    model_data=model_s3_path,
    entry_point='script.py',
    role=role,
    framework_version='1.2-1',
    dependencies=['requirements.txt'],  
    py_version='py3'
)

In [0]:
predictor = predictor.deploy(
    initial_instance_count=1,
    instance_type='ml.c5.xlarge',
    endpoint_name='music-preference-v35'
)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


-

-

-

-

-

-

-

!

In [0]:
#test x-npy input
import numpy as np
predictor.predict(data = np.array([-0.14203581712988783,
 0.08116381116093035,
 -0.03825466221795619,
 -0.0953953487247656,
 -0.06175356672422747,
 -0.012132861981272471,
 0.10634011925307676,
 0.06040216587300194,
 0.034342077856918785,
 -0.043602479315428884,
 0.11817980914868749,
 -0.13974046206244617,
 0.09503260917895681,
 -0.08121639796483002,
 0.1510315508061449,
 0.056179437497172094,
 0.07227743612944067,
 -0.017026299462702753,
 0.009962577512751064,
 -0.011074999011083239,
 -0.06642545319808683,
 -0.08789308800084371,
 0.028115316801050227,
 0.05002132090395155,
 -0.03152040500914182,
 0.06066548477687695,
 -0.14014477210997847,
 0.023650657138504602,
 -0.003950621115132317,
 0.013143107055347148,
 -0.020833723713463276,
 -0.01085559950381946,
 0.11054885349625057,
 0.06444158963562838,
 -0.014901988383480154,
 -0.021626594356767902,
 -0.007579264855426435,
 -0.09455048983726774,
 0.15796827355094095,
 -0.08934561280457613,
 -0.14324474575594603,
 -0.10939191728123027,
 -0.045722925954760636,
 0.00019632197060741476,
 0.0755443829554599,
 -0.010560331690349048,
 -0.03271607640909092,
 0.026546727614711142,
 0.1247281932141331,
 0.04807110531684504,
 0.041939713992179946,
 0.019037920096147944,
 -0.032619179784183876,
 0.04879637657615939,
 0.061404189324479545,
 -0.11977471059259899,
 -0.0932820475497996,
 0.1001690848278384,
 -0.010506757004566035,
 -0.08332342854059921,
 -0.0070328567506996115,
 -0.11686620756887549,
 0.1377037026670155,
 0.08821045808485518,
 -0.10393738422962771,
 0.0961556301141506,
 0.04028432848064235,
 -0.0026879638878170237,
 0.04262110434054672,
 0.05097624546358199,
 -0.05607007078064894,
 0.07822859722711242,
 0.021580983969016074,
 0.14175259233993295,
 0.11929086935519823,
 -0.08210525218784281,
 -0.016054466516650445,
 -0.01819904806263742,
 -0.053282170002705576,
 -0.12434366209402185,
 -0.026304271777099958,
 0.11856139043247785,
 0.09008461006556673,
 -0.023703418865609803,
 0.08497552153013796,
 0.16204843155553514,
 -0.09054273561033087,
 0.07852831824324151,
 0.06671081853420557,
 -0.14494783908381279,
 -0.15140122158938388,
 0.044123252686358895,
 -0.12629070539583312,
 0.13814511547240252,
 0.05519334923125291,
 -0.0942134311870722,
 0.008182554701661807,
 -0.041383353415445835,
 -0.06076065620091846,
 -0.040376189825485985,
 0.031520867592654725,
 -0.05131068474161384,
 0.023572209989737834,
 -0.07603416844525428,
 -0.07302049688894428,
 0.16260407670616384,
 -0.0034385849968103376,
 0.035012074270875285,
 -0.025149090811015466,
 0.07894090955559553,
 -0.03856936309845804,
 -0.1008418725542786,
 -0.06825366917246456,
 0.026497689033674318,
 0.026341909576602367,
 -0.04785181122828215,
 -0.08497802105924072,
 -0.028139379957378256,
 0.12403909128554556,
 -0.08972890699825987,
 -0.08579874135019831,
 -0.06248573638670256,
 0.024677738407410346,
 -0.16407095941898853,
 0.01193091822978736,
 -0.019782146952799347,
 0.15778689364087972,
 0.09999930577135638,
 11.71197302567977]),
 initial_args={
        'ContentType': 'application/x-npy',
        'Accept': 'application/json'
    })

array({'countries': ['Germany', 'Australia', 'Canada', 'United Kingdom', 'United States'], 'probabilities': [0.019101389136249886, 0.026567849518657885, 0.07321074981649198, 0.1832723202432694, 0.5797086470198316]},
      dtype=object)

In [0]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

data = '''
{
  "top_artists": [
    {"artist_name": "Crystal Castles", "playcount": 1034},
    {"artist_name": "Radiohead", "playcount": 972},
    {"artist_name": "Ladytron", "playcount": 831},
    {"artist_name": "Ghostface Killah", "playcount": 801},
    {"artist_name": "UNKLE", "playcount": 722},
    {"artist_name": "DJ Shadow", "playcount": 711},
    {"artist_name": "Buck 65", "playcount": 700},
    {"artist_name": "AFI", "playcount": 67},
    {"artist_name": "Raekwon", "playcount": 59},
    {"artist_name": "Misfits", "playcount": 56},
    {"artist_name": "Johnny Cash", "playcount": 55},
    {"artist_name": "Nine Inch Nails", "playcount": 55},
    {"artist_name": "Bauhaus", "playcount": 55},
    {"artist_name": "ADULT.", "playcount": 53},
    {"artist_name": "DJ Krush", "playcount": 52},
    {"artist_name": "Le Tigre", "playcount": 51},
    {"artist_name": "Daft Punk", "playcount": 499},
    {"artist_name": "Lil' Wayne", "playcount": 498},
    {"artist_name": "M", "playcount": 834},
    {"artist_name": "New Young Pony Club", "playcount": 465},
    {"artist_name": "The Faint", "playcount": 465},
    {"artist_name": "Kanye West", "playcount": 458},
    {"artist_name": "Björk", "playcount": 453},
    {"artist_name": "Black Moon", "playcount": 451},
    {"artist_name": "Aphex Twin", "playcount": 448},
    {"artist_name": "Odd Nosdam", "playcount": 439},
    {"artist_name": "Nas", "playcount": 433},
    {"artist_name": "The Cramps", "playcount": 432},
    {"artist_name": "Oneohtrix Point Never", "playcount": 416},
    {"artist_name": "Boards of Canada", "playcount": 414},
    {"artist_name": "Test Icicles", "playcount": 414},
    {"artist_name": "Drake", "playcount": 411},
    {"artist_name": "MF DOOM", "playcount": 411},
    {"artist_name": "Above the Law", "playcount": 410},
    {"artist_name": "Cansei de Ser Sexy", "playcount": 406},
    {"artist_name": "Air", "playcount": 395},
    {"artist_name": "De La Soul", "playcount": 395},
    {"artist_name": "Gang Starr", "playcount": 390},
    {"artist_name": "Gravediggaz", "playcount": 389},
    {"artist_name": "Ministry", "playcount": 389},
    {"artist_name": "Bone Thugs-N-Harmony", "playcount": 384},
    {"artist_name": "Squarepusher", "playcount": 378},
    {"artist_name": "Grimes", "playcount": 376},
    {"artist_name": "Tricky", "playcount": 373},
    {"artist_name": "The Notorious B.I.G.", "playcount": 371},
    {"artist_name": "Jel", "playcount": 367},
    {"artist_name": "Architecture in Helsinki", "playcount": 360},
    {"artist_name": "Pollie Pop", "playcount": 359},
    {"artist_name": "Lederhosen Lucil", "playcount": 358},
    {"artist_name": "Massive Attack", "playcount": 358}
  ],
  "total_scrobbles": 204985
}
'''
preds = predictor.predict(data = data, 
    initial_args={
        'ContentType': 'application/json',
        'Accept': 'application/json'
    })

In [0]:
preds['countries']

['Australia', 'Germany', 'Canada', 'United Kingdom', 'United States']

In [0]:
preds['probabilities']

[0.030976552048935072,
 0.0359962215528885,
 0.04289745208167965,
 0.20946596067213782,
 0.4201591073620433]

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()